# Step 1 — bf16 reference (Colab Pro, L4)
Runtime → Change runtime type → **L4 GPU**. 🔑 Secrets: `HF_TOKEN`, `WANDB_API_KEY` with notebook access on.
Expected ≈ 2 h ≈ 10 compute units. Resumable: re-run all cells after a disconnect.

In [ ]:
import os
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | cat

In [ ]:
import os, subprocess
REPO = "https://github.com/trinath122/open-nla-audits"
COMMIT = "main"  # pin to a commit hash for the real run
if not os.path.exists("open-nla-audits"):
    subprocess.run(["git", "clone", REPO], check=True)
os.chdir("open-nla-audits")
subprocess.run(["git", "checkout", COMMIT], check=True)
print(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
# keep the platform's CUDA torch; install the rest
!pip -q install -e . --no-deps
!pip -q install "transformers>=5,<6" "accelerate>=1.0" "datasets>=2.20" "huggingface-hub>=0.24" safetensors wandb python-dotenv pyarrow pyyaml

In [ ]:
# Secrets -> environment so the `!python` subprocess inherits them. Values are never printed.
from nla_audit.runtime.env import load_secrets
found = load_secrets(("HF_TOKEN", "WANDB_API_KEY"))
print("secrets found:", found)
assert "HF_TOKEN" in found, "add HF_TOKEN to this platform's secrets"
os.environ.setdefault("HF_ARTIFACT_REPO", "trinath122/open-nla-audits-artifacts")
os.environ.setdefault("WANDB_PROJECT", "open-nla-audits")

In [ ]:
PROFILE = "colab"
!python -m nla_audit.experiments.step1_baseline run --config configs/step1.yaml --profile {PROFILE} --sync

In [ ]:
import json
s = json.load(open(f"runs/step1_baseline/{PROFILE}/summary.json"))
for tag, rep in s["tags"].items():
    print(tag, json.dumps(rep["verdict"], indent=1))
    for k in ("overall", "ufw", "wildchat"):
        r = rep[k]
        print(f"  {k:9s} FVE(own) {r['fve_own']['fve']:.3f} [{r['fve_own']['lo']:.3f}, {r['fve_own']['hi']:.3f}]"
              f"  FVE(0.7335) {r['fve_reference']['fve']:.3f}  parse-fail {r['parse_fail_rate']:.2%}  CJK {r['cjk_any_rate']:.2%}")
    print("  shuffled control:", rep["shuffled_control"])
if "precision_check" in s:
    print("precision check:", json.dumps(s["precision_check"], indent=1))
print("L20 mean norm:", s["mean_norm"]["mean_norm"])
print("monitors:", {k: (v["max_abs"], v["any_nonfinite"]) for k, v in s.get("monitors", {}).items()})

In [ ]:
# After BOTH runs finish: Kaggle fp16 vs Colab bf16 agreement (cos >= 0.999, ΔFVE CI covers 0)
!python -m nla_audit.experiments.step1_baseline compare --a step1_baseline/kaggle:fp16 --b step1_baseline/colab:bf16 --sync